# Cross-species alignment, scored by commutativity

Which human ↔ mouse coupling of the chain sections agrees best with the temporal couplings of each species? This notebook evaluates the output of `bench/scripts/cross_align.py` (couplings, loops and metrics in `mgw/bench/crossloop.py`). It reads only saved files and needs no GPU.

**Rungs:** the cross pairs X1–X10 (`series.yaml`) on the sections of `chains.yaml`. **Loops:** rung pairs where each species advances at most one stage: 6 squares (both advance) and 6 triangles (one species stays, identity on that side).

**Commutativity:** from a tile of the earlier section of species A, *path 1* goes forward in time within A and then across (temporal, then cross); *path 2* goes across and then forward in time within B. Both end on B's later section (row-normalised couplings as Markov kernels). For each start tile, `comm` is the energy distance between the two distributions over the target tiles, in the target's RMS radii. It is 0 when the paths agree, and it credits near misses. The mean is weighted by bins.

**Diffuse maps commute trivially.** A coupling that spreads every tile over the whole target makes both paths uniform. So `comm` is read against `sharp` (each path's distance from the uninformative map) and against the rungs' accuracy (κ_bal both ways, held-out ortholog gene r).

**Methods:**
- ortholog FGW with α ∈ {0 (linear OT), …, 1 (GW on space only)}, balanced (`fgw_a*`) and unbalanced (`ufgw_a*_r*`). The unbalanced runs put a KL penalty ρ on both marginals, in cost units (OTT's τ = ρ/(ρ+ε))
- moscot FGW
- references on the organ-label cost (`label_wot`, `label_fgw_a*`): they use the annotation, so they are shown but never selected

**The rule** (fixed before looking at the results):
1. Only settings whose solves are all usable count. Usable means finite, with the FGW outer loop stopped before its cap (linear OT and moscot use their own convergence flag).
2. The temporal reference is the mean over the DeST-OT temporal runs. Per loop, a setting is *on the front* if no other setting is better by more than `DELTA` (min–max normalised within the loop) in both `comm` and accuracy. Accuracy is the mean normalised κ_bal and gene r of the loop's two rungs. Settings on the front of at least `FRONT_MIN` of the loops are eligible.
3. Among the eligible settings, the lowest mean `comm` wins, among those whose accuracy is within `ACC_TOL` of the best eligible accuracy.
4. **Temporal choice:** under the winning cross setting, the DeST-OT temporal run with the lowest `comm` wins, among those whose temporal κ_bal is within `KAPPA_TOL` of the best one.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
ROOT = None        # bench root; None: $MGW_BENCH_DIR or <data dir>/bench
RES = 5000         # reduce.max_obs of the run
RUN_DIR = None     # results folder <temporal folder>-x<hash>; None: the one with the latest loop table under n<RES>
SEED = 0           # the sweep's seed
DELTA = 0.05       # front tolerance, in min-max units within a loop
FRONT_MIN = 0.75   # share of loops a setting must be on the front of
ACC_TOL = 0.05     # the pick's accuracy (normalised, 0 best) may be this much worse than the best eligible one
KAPPA_TOL = 0.01   # temporal runs within this much of the best temporal kappa compete on commutativity

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

from mgw.bench import crossloop as cl, loopview, metrics, prepare, temporal as tp
from mgw.bench.report.figures import OTHER, SLOTS, organ_palette

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 300)
broot = prepare.bench_root(ROOT)
base = broot / "diag" / "cross" / f"n{RES}"
OUT = base / RUN_DIR if RUN_DIR else max((d for d in base.iterdir() if cl.comm_dir(d, SEED).is_dir()),
                                         key=lambda d: max(f.stat().st_mtime for f in cl.comm_dir(d, SEED).glob("*.csv")))
settings = json.loads((OUT / "settings.json").read_text())
TDIR = broot / "diag" / "temporal" / f"n{RES}" / settings["temporal"]     # the local copy of the temporal folder
rungs = pd.read_csv(OUT / "rungs.csv")
rungs = rungs[rungs.seed == SEED]
loops = pd.read_csv(OUT / "loops.csv")
runs = cl.load_runs(OUT, SEED)
comm = cl.load_comm(OUT, SEED)
final_runs, final_comm = cl.load_final(OUT)
tfinal = tp.load_final(TDIR) if TDIR.is_dir() else None
LOOPS = list(loops.loop)
SQUARES = list(loops.loop[loops.kind == "square"])

INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
SEQ_R = SEQ.reversed()
RHO_MARK = {np.inf: "o", 3.0: "^", 1.0: "s", 0.3: "D", 0.1: "v"}
REF = {"label_wot": (SLOTS[3], "D", "label W-OT (reference)"), "label_fgw": (SLOTS[4], "P", "label FGW (reference)"),
       "moscot": (SLOTS[6], "X", "moscot")}


def style(ax):
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0, labelsize=8)
    for s in ax.spines.values():
        s.set_visible(False)


def grid_axes(n, ncol=4, w=3.4, h=3.0):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, list(axes.flat)


for c in ("error", "outer_stopped", "converged", "ugw_fallback"):
    if c not in runs:
        runs[c] = np.nan
runs["ok"] = runs.error.isna() & runs.finite.fillna(False).astype(bool) & \
    runs.outer_stopped.fillna(runs.converged).fillna(False).astype(bool)
n_rungs = len(rungs)
setting_ok = runs.groupby("run").ok.sum() == n_rungs
comm = comm[comm.error.isna()] if "error" in comm else comm
comm["ok"] = comm.run.map(setting_ok).fillna(False)
runs["rho"] = runs.rho.fillna(np.inf)
info = runs.drop_duplicates("run").set_index("run")[["method", "alpha", "rho", "reference"]]
DESTOT = sorted(t for t in comm.temporal.unique() if t.startswith("destot"))
print(f"{OUT}\n{n_rungs} rungs, {len(LOOPS)} loops ({len(SQUARES)} squares), {runs.run.nunique()} settings, "
      f"{(~setting_ok).sum()} with an unusable solve; temporal runs: {sorted(comm.temporal.unique())}")

## 1. Rungs and loops

In [ ]:
display(rungs.drop(columns="seed").set_index("rung"))
display(loops.set_index("loop"))

## 2. Solver health

Per method: how many solves are usable. FGW counts as usable when the outer loop stopped before its cap. Also shown: how often the unbalanced solve fell back to no unbalanced correction (`ugw_fallback`, a NaN retry), the mass kept, and the share of tiles an unbalanced coupling drops.

In [ ]:
health = runs.groupby("method").agg(settings=("run", "nunique"), solves=("run", "size"), usable=("ok", "mean"),
                                    converged=("converged", "mean"), fallback=("ugw_fallback", "mean"),
                                    mass=("mass", "mean"), dropped_h=("dropped_h", "mean"),
                                    dropped_m=("dropped_m", "mean"), seconds=("seconds_total", "median"))
display(health.round(3))
bad = runs[~runs.ok]
if len(bad):
    print("unusable solves:")
    display(bad[["run", "rung", "error", "converged", "outer_stopped", "n_outer"]].head(40))

## 3. Rung accuracy against α

Mean over the rungs of κ_bal (both directions) and held-out ortholog gene r, one line per marginal penalty ρ (balanced = ∞; unbalanced runs destroy mass where moving it costs more than about ρ). The last panel shows the mass the coupling keeps. The references and moscot are horizontal lines.

In [ ]:
acc = runs[runs.ok].groupby("run").agg(kappa=("kappa", "mean"), gene_r=("gene_r", "mean"), diag=("diag_mass", "mean"),
                                       mass=("mass", "mean")).join(info)
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, col, lab in zip(axes, ("kappa", "gene_r", "diag", "mass"), ("κ_bal (both ways)", "gene r (held-out orthologs)",
                                                                     "same-organ mass", "mass kept")):
    o = acc[acc.method.isin(["fgw", "ufgw"])]
    rhos = sorted(o.rho.unique(), reverse=True)
    for k, t in enumerate(rhos):
        g = o[o.rho == t].sort_values("alpha")
        ax.plot(g.alpha, g[col], marker=RHO_MARK.get(t, "o"), color=SEQ(0.35 + 0.65 * k / max(len(rhos) - 1, 1)),
                lw=1.4, ms=5, label="balanced" if np.isinf(t) else f"ρ = {t:g}")
    for m, (c, mk, lab_) in REF.items():
        for _, r in acc[acc.method == m].iterrows():
            ax.axhline(r[col], color=c, lw=1, ls="--" if m != "moscot" else ":", alpha=0.8)
            ax.text(1.0, r[col], f" {r.name}", color=c, fontsize=7, va="center", transform=ax.get_yaxis_transform())
    ax.set_xlabel("α (GW weight)", fontsize=8)
    ax.set_title(lab, fontsize=9, color=INK)
    style(ax)
axes[0].legend(frameon=False, fontsize=7)
fig.tight_layout()
plt.show()

## 4. Commutativity

Mean `comm` over the loops, both directions and the DeST-OT temporal runs, on an α × ρ grid (balanced: ρ = ∞). Also shown: `sharp` (how far the composed maps are from the uninformative map; higher means sharper) and `comm_rel` = comm / sharp. Squares and triangles are shown separately.

In [ ]:
c_ref = comm[comm.temporal.isin(DESTOT)].copy()
c_ref["group"] = np.where(c_ref.kind == "square", "squares", "triangles")


def heat(ax, s, title, cmap, fmt="{:.2f}"):
    s = s.join(info)
    s = s[s.method.isin(["fgw", "ufgw"])]
    A = sorted(s.alpha.unique())
    T = sorted(s.rho.unique(), reverse=True)
    M = np.full((len(T), len(A)), np.nan)
    for _, r in s.iterrows():
        M[T.index(r.rho), A.index(r.alpha)] = r.iloc[0]
    im = ax.imshow(M, cmap=cmap, aspect="auto")
    for i in range(len(T)):
        for j in range(len(A)):
            if np.isfinite(M[i, j]):
                v = (M[i, j] - np.nanmin(M)) / max(np.nanmax(M) - np.nanmin(M), 1e-12)
                ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if v > 0.55 else INK)
    ax.set_xticks(range(len(A)), [f"{a:g}" for a in A])
    ax.set_yticks(range(len(T)), ["bal." if np.isinf(t) else f"{t:g}" for t in T])
    ax.set_xlabel("α", fontsize=8)
    ax.set_ylabel("ρ", fontsize=8)
    ax.set_title(title, fontsize=9, color=INK)
    ax.tick_params(length=0, labelsize=8, colors=MUTED)


fig, axes = plt.subplots(2, 3, figsize=(15, 6.2))
for row, grp in zip(axes, ("squares", "triangles")):
    g = c_ref[c_ref.group == grp].groupby("run")
    heat(row[0], g.comm.mean().to_frame(), f"comm, {grp} (lower: commutes)", SEQ)
    heat(row[1], g.sharp.mean().to_frame(), f"sharp, {grp} (higher: sharper)", SEQ_R)
    heat(row[2], g.comm_rel.mean().to_frame(), f"comm / sharp, {grp}", SEQ)
fig.tight_layout()
plt.show()

others = c_ref[c_ref.run.map(info.method).isin(list(REF))].groupby("run")[["comm", "sharp", "comm_rel", "tv_organ",
                                                                           "bary", "overlap"]].mean()
print("references and moscot (same averages, all loops):")
display(others.round(3))

In [ ]:
# every loop: settings (by mean comm) x loops, plus the secondary measures of the 10 best settings
per = c_ref.groupby(["run", "loop"]).comm.mean().unstack()[LOOPS]
order = per.mean(axis=1).sort_values().index
fig, ax = plt.subplots(figsize=(11, 0.22 * len(order) + 1.2))
im = ax.imshow(per.loc[order].to_numpy(), cmap=SEQ, aspect="auto")
ax.set_yticks(range(len(order)), order, fontsize=7)
ax.set_xticks(range(len(LOOPS)), [l + (" ▢" if l in SQUARES else " △") for l in LOOPS], fontsize=7, rotation=45)
fig.colorbar(im, ax=ax, label="comm (target RMS radii)", shrink=0.6)
ax.set_title("comm per loop (mean over directions and DeST-OT temporal runs)", fontsize=9, color=INK)
ax.tick_params(length=0, colors=MUTED)
fig.tight_layout()
plt.show()
display(c_ref.groupby("run")[["comm", "sharp", "comm_rel", "tv_organ", "bary", "overlap"]].mean()
        .loc[order[:10]].round(3))

## 5. Selection: commutativity against accuracy

Per loop, `comm` (x) against accuracy (y: mean κ_bal of the loop's two rungs). Colour is α, the marker is ρ, and the references are drawn in their own colours. The grey line marks the loop's Pareto front over the candidates.

In [ ]:
def norm_lower_better(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0


cand_runs = [r for r in setting_ok.index[setting_ok] if not info.loc[r, "reference"]]
rr = runs[runs.ok & runs.run.isin(cand_runs)].copy()
for col in ("kappa", "gene_r"):
    rr[col + "_n"] = rr.groupby("rung")[col].transform(lambda s: norm_lower_better(-s))
rr["acc_n"] = (rr.kappa_n + rr.gene_r_n) / 2
racc = rr.set_index(["run", "rung"])
L = loops.set_index("loop")
rows = []
cl_ = c_ref[c_ref.run.isin(cand_runs)].groupby(["run", "loop"])[["comm", "sharp", "comm_rel"]].mean().reset_index()
for _, r in cl_.iterrows():
    a1, a2 = racc.loc[(r.run, L.loc[r.loop, "r1"])], racc.loc[(r.run, L.loc[r.loop, "r2"])]
    rows.append(dict(r, acc_n=(a1.acc_n + a2.acc_n) / 2, kappa=(a1.kappa + a2.kappa) / 2,
                     gene_r=(a1.gene_r + a2.gene_r) / 2))
sel = pd.DataFrame(rows)
sel["comm_n"] = sel.groupby("loop").comm.transform(norm_lower_better)
sel["front"] = False
for lp, g in sel.groupby("loop"):
    C, A = g.comm_n.to_numpy(), g.acc_n.to_numpy()
    dom = (C[None, :] < C[:, None] - DELTA) & (A[None, :] < A[:, None] - DELTA)
    sel.loc[g.index, "front"] = ~dom.any(1)
summary = sel.groupby("run").agg(loops=("loop", "nunique"), front_share=("front", "mean"), comm=("comm", "mean"),
                                 comm_n=("comm_n", "mean"), acc_n=("acc_n", "mean"), kappa=("kappa", "mean"),
                                 gene_r=("gene_r", "mean"), sharp=("sharp", "mean"),
                                 comm_rel=("comm_rel", "mean")).join(info[["method", "alpha", "rho"]])
summary = summary[summary.loops == len(LOOPS)]
eligible = summary[summary.front_share >= FRONT_MIN]
if eligible.empty:
    eligible = summary[summary.front_share >= summary.front_share.quantile(0.75)]
    print(f"no setting is on the front of {FRONT_MIN:.0%} of the loops; using the top quarter by front share")
best_acc = eligible.acc_n.min()
pool = eligible[eligible.acc_n <= best_acc + ACC_TOL].sort_values("comm")
PICK = pool.index[0]
print("eligible settings (lowest comm first; *_n: 0 is best)")
display(eligible.sort_values("comm").round(3))
print(f"selected cross setting: {PICK}")

fig, axes = grid_axes(len(LOOPS))
for ax, lp in zip(axes, LOOPS):
    g = sel[sel.loop == lp].join(info, on="run")
    f = g[g.front].sort_values("comm")
    ax.plot(f.comm, f.kappa, color=MUTED, lw=1, zorder=1)
    for t, mk in RHO_MARK.items():
        d = g[(g.rho == t) & g.method.isin(["fgw", "ufgw"])]
        ax.scatter(d.comm, d.kappa, c=d.alpha, cmap=SEQ, vmin=0, vmax=1, marker=mk, s=22, edgecolor="none", zorder=2)
    d = g[g.method == "moscot"]
    ax.scatter(d.comm, d.kappa, color=REF["moscot"][0], marker="X", s=40, zorder=3)
    # references: same loop, own rows
    refs = c_ref[(c_ref.loop == lp) & c_ref.run.map(info.reference).fillna(False).astype(bool)].groupby("run").comm.mean()
    for r, cv in refs.items():
        k = runs[(runs.run == r) & runs.rung.isin([L.loc[lp, "r1"], L.loc[lp, "r2"]])].kappa.mean()
        m = info.loc[r, "method"]
        ax.scatter([cv], [k], color=REF[m][0], marker=REF[m][1], s=36, alpha=0.6, zorder=3)
    p = g[g.run == PICK]
    ax.scatter(p.comm, p.kappa, s=140, facecolor="none", edgecolor=INK, lw=1.2, zorder=4)
    ax.set_title(f"{lp} ({L.loc[lp, 'kind']})", fontsize=8, color=INK)
    style(ax)
axes[0].set_xlabel("comm (target RMS radii)", fontsize=8)
axes[0].set_ylabel("κ_bal of the two rungs", fontsize=8)
handles = [Line2D([], [], color=MUTED, marker=mk, ls="", label="balanced" if np.isinf(t) else f"ρ = {t:g}")
           for t, mk in RHO_MARK.items()]
handles += [Line2D([], [], color=c, marker=mk, ls="", label=lab) for c, mk, lab in REF.values()]
handles += [Line2D([], [], color=INK, marker="o", mfc="none", ls="", label="pick")]
fig.legend(handles=handles, loc="lower center", ncol=9, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 6. The temporal setting

Under the picked cross setting and the next two in the pool, `comm` per temporal run (mean over loops and directions), next to each temporal run's own scores from `temporal_align.py final` (seed `SEED`). The temporal baselines are shown for contrast. Only DeST-OT runs compete.

In [ ]:
top = list(pool.index[:3])
tc = comm[comm.run.isin(top)].groupby(["temporal", "run"]).comm.mean().unstack()[top]
tc["mean"] = tc.mean(axis=1)
if tfinal is not None:
    tk = tfinal[tfinal.seed == SEED].groupby("run")[["kappa_ba", "gene_r", "migration", "growth_dist_nt"]].mean()
    tc = tc.join(tk.add_prefix("t_"))
display(tc.sort_values("mean").round(3))
dest = tc[tc.index.str.startswith("destot")]
if "t_kappa_ba" in dest:
    dest = dest[dest.t_kappa_ba >= dest.t_kappa_ba.max() - KAPPA_TOL]
T_PICK = dest.sort_values(PICK).index[0]
print(f"selected temporal run: {T_PICK}")

fig, ax = plt.subplots(figsize=(7, 0.4 * len(tc) + 1))
o = tc.sort_values(PICK)
ax.barh(range(len(o)), o[PICK], color=[SLOTS[0] if t.startswith("destot") else OTHER for t in o.index])
ax.set_yticks(range(len(o)), o.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel(f"comm under {PICK}", fontsize=8)
style(ax)
plt.show()

refs_final = []
for m, cond in (("fgw", lambda s: s.method == "fgw"), ("ufgw", lambda s: s.method == "ufgw")):
    s = summary[cond(summary) & (summary.index != PICK)].sort_values(["front_share", "comm"], ascending=[False, True])
    refs_final += list(s.index[:1])
refs_final += [r for r in ("fgw_a0", "fgw_a1", "label_wot") if r in setting_ok.index]
mo = summary[summary.method == "moscot"].sort_values("comm")
refs_final += list(mo.index[:1])
chosen = list(dict.fromkeys([PICK] + refs_final))
print("\nnext, on the GPU server:\n  python bench/scripts/cross_align.py final --runs " + ",".join(chosen) +
      f" --seeds 0,1 --view-temporal {T_PICK},expr_g1_e0.01")

## 7. The loops, interactively

Choose a loop, a direction, a cross setting and a temporal run. Hover over the start section: path 1 (own species first, then across) lights up in **orange**, path 2 (across first) in **blue**. On the target section both colours add up, so where the paths agree the points turn white. Click pins a tile, and the brush sums the tiles within a radius. The toggles switch the organ colours, the per-tile commutativity on the start section, and the common frame (each section rotated or reflected into the start's frame along the couplings).

The files come from `cross_align.py final --view-temporal …` (seed 0).

In [ ]:
if loopview.view_files(OUT):
    loopview.viewer(OUT, loop="X3-X5", run=PICK if "PICK" in dir() else None,
                    temporal=T_PICK if "T_PICK" in dir() else None)
else:
    print(f"no viewer files yet in {OUT / 'final' / 'view'}: run cross_align.py final with --view-temporal")

## 8. Where the paths disagree

For the picked settings: per-tile `comm` on every loop's start section (from human), and the organ flow of both paths. Each flow is the start organ × target organ mass, with rows normalised.

In [ ]:
tdir_ = OUT / "final" / "tiles"
files = {lp: tdir_ / f"{lp}_h" / f"{PICK}__{T_PICK}_seed{SEED}.npz" for lp in LOOPS} if "PICK" in dir() else {}
files = {k: f for k, f in files.items() if f.exists()}
if not files:
    print("no final tiles for the pick yet (cross_align.py final)")
else:
    fig, axes = grid_axes(len(files), ncol=6, w=2.6, h=2.6)
    vmax = np.quantile(np.concatenate([np.load(f)["comm"] for f in files.values()]), 0.97)
    for ax, (lp, f) in zip(axes, files.items()):
        z = np.load(f)
        ax.scatter(z["xy"][:, 0], z["xy"][:, 1], c=z["comm"], cmap=SEQ, vmin=0, vmax=vmax, s=1.5, lw=0)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.axis("off")
        ax.set_title(f"{lp}: {str(z['start'])}\nmean {np.average(z['comm']):.2f}", fontsize=7, color=INK)
    fig.suptitle(f"comm per start tile, {PICK} with {T_PICK}", fontsize=9)
    fig.tight_layout()
    plt.show()

    lp = next(iter(f for f in files if f in SQUARES), next(iter(files)))
    z = np.load(files[lp])
    names = np.asarray(metrics.ORGANS)
    keep = metrics.SCORED & ((z["flow1"].sum(1) + z["flow2"].sum(1)) > 0.01 * z["flow1"].sum())
    keep_t = metrics.SCORED & ((z["flow1"].sum(0) + z["flow2"].sum(0)) > 0.01 * z["flow1"].sum())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, key, lab in zip(axes, ("flow1", "flow2"), ("path 1 (temporal, then cross)", "path 2 (cross, then temporal)")):
        F = metrics.row_normalize(z[key][np.ix_(keep, keep_t)])
        ax.imshow(F, cmap=SEQ, vmin=0, vmax=1)
        ax.set_xticks(range(keep_t.sum()), names[keep_t], rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(keep.sum()), names[keep], fontsize=7)
        ax.set_title(f"{lp} from human: {lab}", fontsize=9, color=INK)
        ax.tick_params(length=0, colors=MUTED)
    fig.tight_layout()
    plt.show()

## 9. Seed stability

`final` runs both Voronoi tilings (seed 0: random_state 42, seed 1: 43). Per run: how much `comm`, κ_bal and gene r move between the seeds.

In [ ]:
if final_comm.empty or final_comm.seed.nunique() < 2:
    print("needs final runs on two seeds")
else:
    k = ["run", "temporal", "loop", "direction"]
    c2 = final_comm.pivot_table(index=k, columns="seed", values="comm").dropna()
    r2 = final_runs.pivot_table(index=["run", "rung"], columns="seed", values=["kappa", "gene_r"]).dropna()
    s0, s1 = sorted(final_comm.seed.unique())[:2]
    stab = pd.DataFrame({
        "comm |d| median": c2.groupby("run").apply(lambda g: (g[s1] - g[s0]).abs().median()),
        "comm r": c2.groupby("run").apply(lambda g: np.corrcoef(g[s0], g[s1])[0, 1]),
        "comm s0": c2.groupby("run")[s0].mean(), "comm s1": c2.groupby("run")[s1].mean(),
        "kappa |d| median": r2.groupby("run").apply(lambda g: (g[("kappa", s1)] - g[("kappa", s0)]).abs().median()),
        "gene r |d| median": r2.groupby("run").apply(lambda g: (g[("gene_r", s1)] - g[("gene_r", s0)]).abs().median()),
    })
    display(stab.round(4))
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    for n, (r, g) in enumerate(c2.groupby("run")):
        ax.scatter(g[s0], g[s1], s=8, color=SLOTS[n % len(SLOTS)], label=r, alpha=0.7, lw=0)
    lim = [c2.min().min(), c2.max().max()]
    ax.plot(lim, lim, color=MUTED, lw=0.8)
    ax.set_xlabel(f"comm, seed {s0}", fontsize=8)
    ax.set_ylabel(f"comm, seed {s1}", fontsize=8)
    ax.legend(frameon=False, fontsize=6)
    style(ax)
    plt.show()